In [1]:
%load_ext autoreload
%autoreload 2

# `Logit` on Orders - Logistic Regression (~1h)

## Select features

🎯 Haydi `wait_time` ve `delay_vs_expected` değişkenlerinin çok `iyi/kötü review`lar üzerindeki etkisini inceleyelim.

👉 `orders` training_set’imizi kullanarak iki adet `multivariate logistic regression` çalıştıracağız:
- `logit_one` → `dim_is_one_star` tahmini için  
- `logit_five` → `dim_is_five_star` tahmini için.

 

In [1]:
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

👉 Dataset’inizi import edin:

In [4]:
import sys
sys.path.insert(0, '/home/ubtuna')
from olist.order import Order
orders = Order().get_training_data(with_distance_seller_customer=True)
print(orders.columns)
orders.head()

Olist class initialized!
Index(['order_id', 'wait_time', 'expected_wait_time', 'delay_vs_expected',
       'order_status', 'dim_is_five_star', 'dim_is_one_star', 'review_score',
       'number_of_items', 'number_of_sellers', 'price', 'freight_value',
       'distance_seller_customer'],
      dtype='object')


,order_id,wait_time,expected_wait_time,delay_vs_expected,order_status,dim_is_five_star,dim_is_one_star,review_score,number_of_items,number_of_sellers,price,freight_value,distance_seller_customer
0,e481f51cbdc54678b7cc49136f2d6af7,8.436574,15.544063,0.0,delivered,0,0,4,1,1,29.99,8.72,18.681711
1,53cdb2fc8bc7dce0b6741e2150273451,13.782037,19.137766,0.0,delivered,0,0,4,1,1,118.70,22.76,861.035367
2,47770eb9100c2d0c44946d9cf07ec65d,9.394213,26.639711,0.0,delivered,1,0,5,1,1,159.90,19.22,514.547140
3,949d5b44dbf5de918fe9c16f97b45f8a,13.208750,26.188819,0.0,delivered,1,0,5,1,1,45.00,27.20,1821.802656
4,ad21c59c0840e6cb83a9ceb5573f8159,2.873877,12.112049,0.0,delivered,1,0,5,1,1,19.90,8.72,29.593095


👉 Kullanmak istediğiniz feature’ları bir listede seçin:

⚠️ Data leakage yaratmadığınızdan emin olun (yani target’tan türetilmiş feature’ları seçmeyin)

💡 `wait_time` ve `delay_vs_expected` değişkenlerinin etkisini anlayabilmek için diğer feature’ların etkisini kontrol etmemiz gerekir, bu yüzden listenize ilgili olabilecek tüm feature’ları dahil edin.

-order_id → sadece identifier, predictive değil

-order_status → kategorik, ve zaten çoğu "delivered" (varyans yok)

-dim_is_five_star, dim_is_one_star, review_score → bunlar bizim target'larımız! Birini tahmin ederken diğerini feature olarak koyarsak = data leakage

In [5]:
features = [
    'wait_time',
    'expected_wait_time',
    'delay_vs_expected',
    'number_of_items',
    'number_of_sellers',
    'price',
    'freight_value',
    'distance_seller_customer'
]


X = orders[features]
y_one = orders['dim_is_one_star']
y_five = orders['dim_is_five_star']

# Hızlı kontrol
print(f"X shape: {X.shape}")
print(f"y_one shape: {y_one.shape} | 1-star oranı: {y_one.mean():.2%}")
print(f"y_five shape: {y_five.shape} | 5-star oranı: {y_five.mean():.2%}")

X.head()

X shape: (95872, 8)
y_one shape: (95872,) | 1-star oranı: 9.77%
y_five shape: (95872,) | 5-star oranı: 59.21%


,wait_time,expected_wait_time,delay_vs_expected,number_of_items,number_of_sellers,price,freight_value,distance_seller_customer
0,8.436574,15.544063,0.0,1,1,29.99,8.72,18.681711
1,13.782037,19.137766,0.0,1,1,118.70,22.76,861.035367
2,9.394213,26.639711,0.0,1,1,159.90,19.22,514.547140
3,13.208750,26.188819,0.0,1,1,45.00,27.20,1821.802656
4,2.873877,12.112049,0.0,1,1,19.90,8.72,29.593095


🕵🏻 Feature’larınızın `multicollinearity` durumunu `VIF index` kullanarak kontrol edin.

* Çok yüksek olmamalıdır (tercihen < 10), böylece partial regression coefficient’larına ve ilgili `p-values` değerlerine güvenebiliriz.
* Verinizi standardize etmeyi unutmayın!
    * Bir `VIF Analysis`, bir feature’ın diğer feature’lara karşı regresyonunu yaparak hesaplanır...
    * Bu yüzden herhangi bir linear regression çalıştırmadan önce feature’ların `scale etkisini kaldırmak` ve eşit öneme sahip olmalarını sağlamak istersiniz!
    
    
📚 <a href="https://www.statisticshowto.com/variance-inflation-factor/">Statistics How To - Variance Inflation Factor</a>

📚  <a href="https://online.stat.psu.edu/stat462/node/180/">PennState - Detecting Multicollinearity Using Variance Inflation Factors</a>

⚖️ Standardize etme:

In [6]:

X_scaled = (X - X.mean()) / X.std()


print(X_scaled.describe().loc[['mean', 'std']].round(3))

      wait_time  expected_wait_time  delay_vs_expected  number_of_items  \
mean        0.0                 0.0                0.0             -0.0   
std         1.0                 1.0                1.0              1.0   

      number_of_sellers  price  freight_value  distance_seller_customer  
mean                0.0    0.0           -0.0                      -0.0  
std                 1.0    1.0            1.0                       1.0  


👉 Olası multicollinearity durumlarını analiz etmek için VIF Analysis’inizi çalıştırın:

In [11]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_df = pd.DataFrame({
    'feature': X_scaled.columns,
    'VIF': [variance_inflation_factor(X_scaled.values, i) for i in range(X_scaled.shape[1])]
}).sort_values('VIF', ascending=False).reset_index(drop=True)

orders_scaled = X_scaled.copy()
orders_scaled['dim_is_one_star'] = orders['dim_is_one_star'].values
orders_scaled['dim_is_five_star'] = orders['dim_is_five_star'].values

formula_features = ' + '.join(features)

vif_df

,feature,VIF
0,wait_time,3.061637
1,delay_vs_expected,2.449058
2,freight_value,1.680201
3,distance_seller_customer,1.603192
4,expected_wait_time,1.602753
5,number_of_items,1.371594
6,price,1.208614
7,number_of_sellers,1.095545


## Logistic Regressions

👉 İki adet `Logistic Regression` modeli fit edin:
- `logit_one` → `dim_is_one_star` tahmini için
- `logit_five` → `dim_is_five_star` tahmini için.

`Logit 1️⃣`

In [12]:
logit_one = smf.logit(
    formula=f'dim_is_one_star ~ {formula_features}',
    data=orders_scaled
).fit()

print(logit_one.summary())

Optimization terminated successfully.
         Current function value: 0.272747
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:        dim_is_one_star   No. Observations:                95872
Model:                          Logit   Df Residuals:                    95863
Method:                           MLE   Df Model:                            8
Date:                Fri, 17 Apr 2026   Pseudo R-squ.:                  0.1474
Time:                        14:37:47   Log-Likelihood:                -26149.
converged:                       True   LL-Null:                       -30669.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                   -2.4842      0.013   -189.092      0.000      -2.510

`Logit 5️⃣`

In [13]:
logit_five = smf.logit(
    formula=f'dim_is_five_star ~ {formula_features}',
    data=orders_scaled
).fit()

print(logit_five.summary())

Optimization terminated successfully.
         Current function value: 0.636467
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:       dim_is_five_star   No. Observations:                95872
Model:                          Logit   Df Residuals:                    95863
Method:                           MLE   Df Model:                            8
Date:                Fri, 17 Apr 2026   Pseudo R-squ.:                 0.05859
Time:                        14:37:53   Log-Likelihood:                -61019.
converged:                       True   LL-Null:                       -64817.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                    0.3416      0.007     47.749      0.000       0.328

💡 Şimdi bu iki logistic regression’ın sonuçlarını analiz etme zamanı:

- Partial coefficient’ları kendi kelimelerinizle yorumlayın.
- `p-values` kullanarak istatistiksel anlamlılıklarını kontrol edin.
- Coefficient önemleri açısından `logit_one` ve `logit_five` arasında herhangi bir fark görüyor musunuz?

In [ ]:
# Aşağıdaki cümlelerden doğru olanları aşağıdaki listeye kaydedin.

a = "delay_vs_expected influences five_star ratings even more than one_star ratings"
b = "wait_time influences five_star ratings even more than one_star"

your_answer = [a]

#0.375 > 0.139  5-star üzerindeki etki daha büyük gözüküyor. Bu yüzden a doğru bir önerme evet delay_vs_expected'ın 5-star değerlendirmeleri üzerinde 1-star değerlendirmelerine göre daha büyük bir etkisi var gibi görünüyor. Ancak, bu tür sonuçları yorumlarken dikkatli olmak önemlidir, çünkü modeldeki diğer faktörler ve veri setinin özellikleri de etkili olabilir.

#wait_time her iki modelde de güçlü ama 1-star'ı daha çok etkiliyor çünkü çok uzun bekleyen müşteri doğrudan şikayete gidiyor → 1 yıldız basıyor.
#delay_vs_expected ise beklenti ihlali → müşteri "5 Nisan'da gelir" diye planlamış, 10'unda gelince "ay beklediğim gibi değildi, 5 verememem" diyor ama şikayet edecek kadar da sinirlenmiyor → 5-star'dan düşüyor ama 1-star'a inmiyor.
#Yani delay, beklentiyle oynuyor, wait_time ise genel tatmin düzeyini düşürüyor.

🧪 __Kodunu Test Et__

In [16]:
from nbresult import ChallengeResult

result = ChallengeResult('logit',
    answers = your_answer
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/ubtuna/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/ubtuna/data-logit/tests
plugins: anyio-4.8.0, typeguard-4.4.2, dash-4.1.0
collecting ... collected 1 item

test_logit.py::TestLogit::test_question PASSED                           [100%]

============================== 1 passed in 0.02s ===============================


💯 You can commit your code:

git add tests/logit.pickle

git commit -m 'Completed logit step'

git push origin master



<details>
    <summary>- <i>Açıklamalar ve ileri seviye kavramlar</i> -</summary>


> _Diğer tüm şeyler sabitken, `delay factor`, 1-yıldız review alma ihtimalini etkilemesinden bile daha fazla, 5-yıldızdan mahrum kalma ihtimalini artırma eğilimindedir. Muhtemelen bunun sebebi, 1-yıldız review’ların bizzat çok kötü ürünleri hedeflemesi, kötü teslimatları değil._

❗️ Ancak tamamen titiz olmak için, **iki farklı modelin coefficient’larını karşılaştırırken daha dikkatli olmamız gerekir**, çünkü **benzer popülasyonlara dayanmayabilirler**!
    Burada 2 alt popülasyonumuz var: (1-yıldız verenler ve 5-yıldız verenler) ve bunlar doğaları gereği farklı davranış kalıpları sergileyebilirler. 5-yıldız vermeye daha meyilli “mutlu insanlar”ın, “gecikme” veya “fiyat” söz konusu olduğunda, 1-yıldızı “Lucky-Luke gibi ateşleyen” “huysuz insanlara” göre daha az hassas olmaları gayet mümkün...

</details>



🏁 Tebrikler!

💾 `logit.ipynb` notebook’unuzu commit ve push etmeyi unutmayın!